# Build 20-resume corpus

Loads [data/dam_resume.csv](data/dam_resume.csv) as **ResumeID 1**, merges 19 synthetic personas from [data/synthetic_personas.json](data/synthetic_personas.json), and writes:

- `data/corpus_20_resumes.csv` — same schema as `dam_resume.csv`
- `data/resume_metadata.csv` — one row per resume for benchmarking similarity

Run **Kernel → Restart & Run All** after changing personas or paths. Working directory should be `peopleLikeMe` (or repo root; paths auto-resolve).

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd


def resolve_data_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "dam_resume.csv").exists():
        return cwd / "data"
    if (cwd / "peopleLikeMe" / "data" / "dam_resume.csv").exists():
        return cwd / "peopleLikeMe" / "data"
    raise FileNotFoundError(
        "Could not find data/dam_resume.csv; cwd=" + str(cwd)
    )


DATA_DIR = resolve_data_dir()
DAM_RESUME_PATH = DATA_DIR / "dam_resume.csv"
PERSONAS_PATH = DATA_DIR / "synthetic_personas.json"
CORPUS_PATH = DATA_DIR / "corpus_20_resumes.csv"
META_PATH = DATA_DIR / "resume_metadata.csv"

DATA_DIR, DAM_RESUME_PATH.exists(), PERSONAS_PATH.exists()

(WindowsPath('C:/Users/DeloresMincarelli/Documents/Portfolio/peopleLikeMe/data'),
 True,
 True)

In [2]:
def rows_for_persona(p: dict) -> list[dict]:
    """Long-form rows matching dam_resume.csv schema. ItemID restarts at 1 per resume."""
    rid = int(p["resume_id"])
    rows: list[dict] = []
    i = 1
    rows.append(
        {
            "ResumeID": rid,
            "Section": "Summary",
            "Title": p["summary_title"],
            "ItemID": i,
            "Text": p["summary_text"],
            "MonthsDuration": pd.NA,
            "RecencyMonthsAgo": pd.NA,
        }
    )
    i += 1
    for exp in p["experience"]:
        rows.append(
            {
                "ResumeID": rid,
                "Section": "Experience",
                "Title": exp["title"],
                "ItemID": i,
                "Text": exp["text"],
                "MonthsDuration": int(exp["months"]),
                "RecencyMonthsAgo": int(exp["recency"]),
            }
        )
        i += 1
    for idx, edu in enumerate(p["education"]):
        rec = 36 + idx * 48
        rows.append(
            {
                "ResumeID": rid,
                "Section": "Education",
                "Title": pd.NA,
                "ItemID": i,
                "Text": edu,
                "MonthsDuration": pd.NA,
                "RecencyMonthsAgo": rec,
            }
        )
        i += 1
    for sk in p["skills"]:
        rows.append(
            {
                "ResumeID": rid,
                "Section": "Skills",
                "Title": pd.NA,
                "ItemID": i,
                "Text": sk,
                "MonthsDuration": pd.NA,
                "RecencyMonthsAgo": pd.NA,
            }
        )
        i += 1
    return rows


def load_personas() -> list[dict]:
    with open(PERSONAS_PATH, encoding="utf-8") as f:
        return json.load(f)

In [3]:
dam = pd.read_csv(DAM_RESUME_PATH, dtype={"ResumeID": int, "ItemID": int})
personas = load_personas()
synth_rows: list[dict] = []
for p in personas:
    synth_rows.extend(rows_for_persona(p))

synth = pd.DataFrame(synth_rows)
for col in ("MonthsDuration", "RecencyMonthsAgo"):
    synth[col] = pd.to_numeric(synth[col], errors="coerce").astype("Int64")

corpus = pd.concat([dam, synth], ignore_index=True)
corpus["ResumeID"] = corpus["ResumeID"].astype(int)
corpus["ItemID"] = corpus["ItemID"].astype(int)
corpus["Title"] = corpus["Title"].astype(object).where(corpus["Title"].notna(), "")

corpus.to_csv(CORPUS_PATH, index=False, encoding="utf-8", na_rep="")

meta_me = {
    "ResumeID": 1,
    "primary_domain": "data_analytics",
    "role_family": "healthcare_analytics",
    "seniority_band": "mid",
}
meta_rest = [
    {
        "ResumeID": int(p["resume_id"]),
        "primary_domain": p["primary_domain"],
        "role_family": p["role_family"],
        "seniority_band": p["seniority_band"],
    }
    for p in personas
]
meta = pd.DataFrame([meta_me] + meta_rest).sort_values("ResumeID").reset_index(drop=True)
meta.to_csv(META_PATH, index=False, encoding="utf-8")

len(corpus), corpus["ResumeID"].nunique(), CORPUS_PATH, META_PATH

(236,
 20,
 WindowsPath('C:/Users/DeloresMincarelli/Documents/Portfolio/peopleLikeMe/data/corpus_20_resumes.csv'),
 WindowsPath('C:/Users/DeloresMincarelli/Documents/Portfolio/peopleLikeMe/data/resume_metadata.csv'))

In [4]:
reread = pd.read_csv(CORPUS_PATH, dtype={"ResumeID": int, "ItemID": int})
assert reread["ResumeID"].nunique() == 20
assert len(meta) == 20
assert set(meta["ResumeID"]) == set(range(1, 21))
missing = set(reread["ResumeID"].unique()) ^ set(range(1, 21))
assert not missing, missing

reread.groupby("ResumeID").size().describe()

count    20.00000
mean     11.80000
std       3.59239
min      10.00000
25%      11.00000
50%      11.00000
75%      11.00000
max      27.00000
dtype: float64